In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import random

TICKER = "AAPL"
df = pd.read_parquet(f"../data/raw/earnings/{TICKER}.parquet")

# Normalize to ET
df["earnings_date"] = pd.to_datetime(df["earnings_date"], utc=True).dt.tz_convert("America/New_York")
df["hour"] = df["earnings_date"].dt.hour + df["earnings_date"].dt.minute / 60

df.head(10)

In [ ]:
def bucket(hour: float) -> str:
    if hour < 9.5:
        return "BMO"
    elif hour < 16.0:
        return "Intraday"
    elif hour == 16.0:
        return "At close (16:00)"
    else:
        return "AMC (>16:00)"

df["bucket"] = df["hour"].apply(bucket)

ORDER = ["BMO", "Intraday", "At close (16:00)", "AMC (>16:00)"]
counts = df["bucket"].value_counts().reindex(ORDER, fill_value=0)
print(counts)
df[["earnings_date", "hour", "bucket"]].sort_values("hour")

In [ ]:
COLORS = {
    "BMO": "#4C72B0",
    "Intraday": "#DD8452",
    "At close (16:00)": "#55A868",
    "AMC (>16:00)": "#C44E52",
}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle(f"{TICKER} — Earnings Release Time Distribution (ET)", fontsize=13)

# Left: bar chart
ax = axes[0]
bars = ax.bar(counts.index, counts.values, color=[COLORS[b] for b in counts.index], edgecolor="white", linewidth=0.8)
ax.bar_label(bars, padding=3, fontsize=11)
ax.set_ylabel("Number of releases")
ax.set_title("Releases by bucket")
ax.set_ylim(0, counts.max() * 1.18)
ax.spines[["top", "right"]].set_visible(False)

# Right: strip plot of raw hours
ax2 = axes[1]
for _, row in df.iterrows():
    ax2.scatter(
        row["hour"] + random.uniform(-0.08, 0.08),
        0,
        color=COLORS[row["bucket"]],
        alpha=0.7, s=60, zorder=3,
    )

for xval, label in [(9.5, "Open 09:30"), (16.0, "Close 16:00")]:
    ax2.axvline(xval, color="black", linestyle="--", linewidth=1, alpha=0.5)
    ax2.text(xval + 0.1, 0.35, label, fontsize=8)

ax2.set_xlim(0, 24)
ax2.set_xticks(range(0, 25, 2))
ax2.set_xticklabels([f"{h:02d}:00" for h in range(0, 25, 2)], rotation=45, ha="right")
ax2.set_yticks([])
ax2.set_title("Raw release hours (each dot = one earnings event)")
ax2.set_xlabel("Hour (ET)")
ax2.spines[["top", "right", "left"]].set_visible(False)

legend_patches = [mpatches.Patch(color=c, label=b) for b, c in COLORS.items()]
ax2.legend(handles=legend_patches, loc="upper left", fontsize=9)

plt.tight_layout()
plt.show()